# Setup

In [ ]:
import sys
import pandas as pd
import numpy as np
import json
from pathlib import Path
from datetime import datetime


In [ ]:
COHORT_NAME = "male_50-54"
OUTPUT_DIR  = Path(f"/XBoost_results/{COHORT_NAME}")


# Load Artifacts from run.ipynb

In [ ]:
import joblib
import pandas as pd
import numpy as np

# Load trained model and data saved by run.ipynb
model5_best_model = joblib.load(OUTPUT_DIR / f"model5_raw_pipeline_{COHORT_NAME}.joblib")

X_test_raw  = pd.read_parquet(OUTPUT_DIR / f"X_test_raw_{COHORT_NAME}.parquet")
y_test      = pd.read_parquet(OUTPUT_DIR / f"y_test_raw_{COHORT_NAME}.parquet")["y_test"]
X_cal_raw   = pd.read_parquet(OUTPUT_DIR / f"X_cal_raw_{COHORT_NAME}.parquet")
y_cal       = pd.read_parquet(OUTPUT_DIR / f"y_cal_raw_{COHORT_NAME}.parquet")["y_cal"]

# Load threshold and id_test from saved predictions
preds       = pd.read_parquet(OUTPUT_DIR / "predictions.parquet")
model5_thr  = float(preds["threshold"].iloc[0])
id_test     = preds["pnr"]

# Calibrated Model
## Fit and Calibration Curve

In [ ]:
from sklearn.calibration import CalibratedClassifierCV
calibrated_model = CalibratedClassifierCV(
    estimator=model5_best_model,
    method="isotonic",
    cv= "prefit"
)
calibrated_model.fit(X_cal_raw,y_cal)

In [ ]:
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt
#Raw model
y_prob_raw = model5_best_model.predict_proba(X_test_raw)[:,1]
# Calibrated model
y_prob_cal = calibrated_model.predict_proba(X_test_raw)[:,1]
prob_true_raw, prob_pred_raw=calibration_curve(y_test,y_prob_raw, n_bins=10)
prob_true_cal, prob_pred_cal=calibration_curve(y_test,y_prob_cal, n_bins=10)
plt.figure(figsize=(6,6))
plt.plot(prob_pred_raw, prob_true_raw, marker="o", label="Raw model")
plt.plot(prob_pred_cal, prob_true_cal, marker="o", label="Calibrated model")
plt.plot([0,1],[0,1], linestyle="--", color="gray", label="Perfect ↪ calibration")
plt.xlabel("Predicted probability")
plt.ylabel("Observed probability")
plt.title("Calibration Curve")
plt.legend()
plt.savefig( OUTPUT_DIR/ f"calibration_curve_{COHORT_NAME}.png", dpi=300,
↪ bbox_inches="tight") plt.show()

In [ ]:

from sklearn.metrics import precision_recall_curve, average_precision_score
import matplotlib.pyplot as plt
precision, recall, threshold = precision_recall_curve(y_test,y_prob)
pr_auc=average_precision_score(y_test,y_prob)
plt.plot(threshold,precision[:-1], label="Precision")
plt.plot(threshold,recall[:-1], label= "Recall")
plt.axvline(
    x=model5_thr,
    color="red",
        linestyle="--",
    linewidth=2,
    #label=f"Chosen threshold ={model5_thr:.2f} "
)
plt.scatter(model5_thr, pr_at_thr, color="red",s=100,zorder=6)
plt.scatter(model5_thr, rec_at_thr, color="red",s=100,zorder=6)
plt.hlines(
    y=pr_at_thr,
    xmin=-0,
    xmax=model5_thr,
    colors="red",
    linestyles="dashed",
    linewidth=2
)
plt.hlines(
    y=rec_at_thr,
    xmin=0,
    xmax=model5_thr,
    colors="red",
    linestyles="dashed",
    linewidth=2
)
textstr= (
    f"Threshold={model5_thr:.2f}\n"
    f"Recall={rec_at_thr:.2f}\n"
    f"Precision={pr_at_thr:.2f}"
)
plt.text(
    0.6,0.8,
    textstr,
    fontsize=10,
    bbox=dict(
        boxstyle="round,pad=0.3",
        facecolor="white",
        edgecolor="black",
        alpha=0.9
)
)
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.legend(loc="center left", frameon=True)
plt.grid()
plt.title(f"Precision, recall vs threshold, testdata,{COHORT_NAME}")
plt.savefig(OUTPUT_DIR/ f"precision_recall_threshold_{COHORT_NAME}.png",
↪ dpi=300, bbox_inches="tight") plt.show

In [ ]:
#Find optimal calibtaion threshold
from sklearn.metrics import precision_recall_curve
import numpy as np
y_prob_cal = calibrated_model.predict_proba(X_cal_raw)[:, 1]
precision_cal, recall_cal, thresholds_cal=
↪ precision_recall_curve(y_cal,y_prob_cal)
# Choose threshold that maximizes f2 on calibtaion set
beta=2
p_cal= precision_cal[:-1]
r_cal= recall_cal[:-1]
f2_cal = (1 + beta**2) * (p_cal *r_cal) / (beta**2 * p_cal + r_cal + 1e-12)
best_idx_cal = np.argmax(f2_cal)
model5_thr_cal = float(thresholds_cal[best_idx_cal])
print("Chosen threshold ( max f2):", model5_thr_cal)
print("Precision / Recall at chosen thr:", p_cal[best_idx_cal],
↪ r_cal[best_idx_cal])
print("F2 at chosen thr:", f2_cal[best_idx_cal])

## Evaluation

In [ ]:
## Predicted probability for survivors vs deaths + gap. Calibrated model.
    # split data by outcome
X_survivors = X_test_raw[y_test == 0].copy()
X_deaths = X_test_raw[y_test == 1].copy()
#### OBS model
    #  gap
p_survivors_cal = calibrated_model.predict_proba(X_survivors)[:, 1].mean()
p_deaths_cal = calibrated_model.predict_proba(X_deaths)[:, 1].mean()
total_gap_cal = p_deaths_cal - p_survivors_cal
print("=== Predicted probabilites on test Calibrated model ===")
print(f"Survivors (y=0), mean p(death): {p_survivors_cal:.4f}")
print(f"Early deaths (y=1), mean p(death): {p_deaths_cal:.4f}")
print(f"Mortality gap (death-survivors): {total_gap_cal:.4f}")

In [ ]:
# Evaluation on calibration set
y_prob_cal=calibrated_model.predict_proba(X_cal_raw)[:,1]
y_pred_cal = (y_prob_cal >=model5_thr_cal).astype(int)
cal_f1= f1_score(y_cal, y_pred_cal)
cal_f2 =fbeta_score(y_cal,y_pred_cal,beta=2)
cal_prec=precision_score(y_cal,y_pred_cal)
cal_rec=recall_score(y_cal,y_pred_cal)
cal_roc=roc_auc_score(y_cal,y_prob_cal)
cal_pr=average_precision_score(y_cal,y_prob_cal)
cal_bal=balanced_accuracy_score(y_cal,y_pred_cal)
cal_acc=accuracy_score(y_cal,y_pred_cal)
cal_mcc= matthews_corrcoef(y_cal, y_pred_cal)
print("\n ==== Calibration metric=====" )
print("F1 score:", cal_f1)
print("F2 score:", cal_f2)
print("Precision:", cal_prec)
print("Recall:", cal_rec)
print("ROC-AUC", cal_roc)
print("PR-AUC (avg prec):", cal_pr)
print("Balanced accuracy:", cal_bal)
print("Accuracy:", cal_acc)
print("MCC:", cal_mcc)

In [ ]:
from sklearn.metrics import confusion_matrix
tn_cal,fp_cal, fn_cal, tp_cal=confusion_matrix(y_cal,y_pred_cal).ravel()
cal_specificity = tn_cal/ (tn_cal+ fp_cal)
cal_fpr =fp_cal /(fp_cal+ tn_cal)

In [ ]:
# Calibrated predection
from pathlib import Path
import pandas as pd
import numpy as np
y_prob_cal=calibrated_model.predict_proba(X_test_raw)[:,1]
y_pred_cal = (y_prob_cal >=model5_thr_cal).astype(int)
pred_cal_df = pd.DataFrame({
    "pnr":id_test.astype(str),
    "y_test":np.asarray(y_test).astype(int),
    "y_proba_cal":y_prob_cal,
    "y_pred":y_pred_cal,
    "celibrated_threshold":model5_thr_cal,
    "test_death_rate":y_test.mean()
})
pred_cal_df.to_parquet(OUTPUT_DIR/ "calibrated_predictions.parquet", index=False)

# Save Results

In [ ]:
import joblib
joblib.dump(calibrated_model,
    OUTPUT_DIR / f"model5_calibrated_model_{COHORT_NAME}.joblib")

In [ ]:
# Save calibration data after preprocessing
preprocessor_fitted = model5_best_model.named_steps["preprocess"]
X_cal_processed = preprocessor_fitted.transform(X_cal_raw)
if hasattr(X_cal_processed, "toarray"): X_cal_processed = X_cal_processed.toarray()
X_cal_processed = pd.DataFrame(
    X_cal_processed,
    columns=preprocessor_fitted.get_feature_names_out()
)
X_cal_processed.columns = (
    X_cal_processed.columns
    .str.replace("^remainder__", "", regex=True)
    .str.replace("^cat__", "", regex=True)
)
X_cal_processed.to_csv(OUTPUT_DIR / f"X_cal_{COHORT_NAME}.csv", index=False)
pd.DataFrame({"y_cal": y_cal}).to_csv(OUTPUT_DIR / ↪ f"y_cal_{COHORT_NAME}.csv", index=False)


In [ ]:
# Save results
import numpy as np
time_stamp=datetime.now().strftime("%Y%m%d_%H%M%S")
results = {
    "model_name": "Model5",
    "objective":{
        "cohort": f"{COHORT_NAME}",
        "maximize":"f2",
        "eval_metric":"aucpr",
        "min_precision": 0,
    "cv_folds":3,
    "n_iter":60,
    "chosen threshold":float(model5_thr),
    "scale_pos_weight":float(spw),
    #"theshold_source": "OOF_train",
    "comments": "Class imbalanced handled with scale_pos_weight instead
↪ of rebalancing" },
"best_params": model_5_best_params,
"death rate summary" :{
    "raw_train_death_rate": raw_train_death_rate,
    "test_death_rate": test_death_rate,
    "calibration death rate": cal_death_rate,
    #"resampled_train_death_rate": resampled_train_death_rate,
},
"test_metrics": {
    "f1": float(test_f1),
    "f2": float(test_f2),
    "precision": float(test_prec),
    "recall":float(test_rec),
    "roc_auc":float(test_roc),
    "pr_auc": float(test_pr),
    "specificity (TNR)": float(specificity),
    "balanced accuracy": float(test_bal),
    "accuracy": float(test_acc),
    "MCC":float(test_mcc),
    "threshold_used": float(chosen_threshold),
},
    "train_metrics": {
    "f1": float(train_f1),
    "f2": float(train_f2),
    "precision": float(train_prec),
    "recall":float(train_rec),
    "roc_auc":float(train_roc),
    "pr_auc": float(train_pr),
    "balanced accuracy": float(train_bal),
    "accuracy": float(train_acc),
    "MCC":float(train_mcc),
    "threshold_used": float(model5_thr),
},
    "calibration_metrics": {
     "f1": float(cal_f1),
     "f2": float(cal_f2),
     "precision": float(cal_prec),
     "recall":float(cal_rec),
     "roc_auc":float(cal_roc),
     "pr_auc": float(cal_pr),
     "balanced accuracy": float(cal_bal),
     "accuracy": float(cal_acc),
     "threshold_used": float(model5_thr_cal),
     "MCC":float(cal_mcc),
     "specificity": float(cal_specificity),
     "fpr":float(cal_fpr),
},
 "confusion_matrix_test":{
     "TN":int(tn),
     "FP": int(fp),
     "FN": int(fn),
     "TP":int(tp),
},
 "Predicted mortality rate test": {
     "overall_predicted_mortality rate": float(predicted_mortality_rate),
},
 "predicted_probabilites test":{
     "mean_survived":float(np.mean(y_prob[y_test==0])),
     "mean_died":float(np.mean(y_prob[y_test==1])),
     "gap_died_minus_survived": float(
     np.mean(y_prob[y_test==1])-np.mean(y_prob[y_test==0])
     )
},
 "predicted_probabilites_calibration":{
     "mean_survived":float(np.mean(y_prob_cal[y_cal==0])),
     "mean_died":float(np.mean(y_prob_cal[y_cal==1])),
     "gap_died_minus_survived": float(
     np.mean(y_prob_cal[y_cal==1])-np.mean(y_prob_cal[y_cal==0])
    ), },
}
out_path =OUTPUT_DIR/ f"Model_5_{COHORT_NAME}{time_stamp}.json"
with open(out_path, "w") as f: json.dump(results,f,indent=2)
print("Saved JSON:", out_path)